In [1]:
import pandas as pd

import torch
from torch import nn

In [2]:
class Dataset:
    def __init__(
        self, 
        X_train: torch.Tensor, 
        Y_train: torch.Tensor,
        X_val: torch.Tensor | None = None,
        Y_val: torch.Tensor | None = None,
        batch_size: int = 32
    ) -> None:
        self.batch_size = batch_size
        
        self.train_loader = self.get_tensorloader(X_train, Y_train, True)
        self.val_loader = None
        if X_val is not None and Y_val is not None:
            self.val_loader = self.get_tensorloader(X_val, Y_val, False)

    def get_tensorloader(self, X: torch.Tensor, Y: torch.Tensor, shuffle=False) -> torch.utils.data.DataLoader:
        tensor_dataset = torch.utils.data.TensorDataset(X, Y)
        
        return torch.utils.data.DataLoader(tensor_dataset, self.batch_size, shuffle)

In [3]:
def accuracy(
    predictions: torch.Tensor,
    targets: torch.Tensor,
) -> float:
    predictions = predictions.reshape(-1)
    targets = targets.reshape(-1)

    if predictions.numel() != targets.numel():
        raise ValueError("Predictions and targets must have equal sizes.")

    return (predictions == targets).float().mean().item()

In [4]:
class LinearClassifier(nn.Module):
    def __init__(self, in_features: int) -> None:
        super().__init__()

        self.linear = nn.Linear(in_features=in_features, out_features=1)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return self.linear(X)

    @torch.no_grad()
    def predict(self, x: torch.Tensor) -> torch.Tensor:
        logits = self(x)
        
        return (logits >= 0).long().squeeze(-1)

In [5]:
class Trainer:
    def __init__(
        self,
        loss_function: nn.Module,
        optimizer: torch.optim.Optimizer,
        max_epochs: int = 100,
    ) -> None:
        self.loss_function = loss_function
        self.optimizer = optimizer
        self.max_epochs = max_epochs

        self.device = torch.device(
            'cuda' if torch.cuda.is_available() else 'cpu'
        )

    def prepare_model(self, model):
        self.model = model.to(self.device)

    def prepare_batch(
        self,
        batch: tuple[torch.Tensor, torch.Tensor],
    ) -> tuple[torch.Tensor, torch.Tensor]:
        X, Y = batch
        
        return X.to(self.device), Y.to(self.device)
    
    def fit(self, model: nn.Module, data: Dataset):
        self.prepare_model(model)
        
        train_loader = data.train_loader
        val_loader = data.val_loader
        
        train_losses = []
        val_losses = []
        val_accs = []
        for epoch in range(self.max_epochs):
            self.model.train()

            train_loss = 0.0
            for batch in train_loader:
                X_batch, Y_batch = self.prepare_batch(batch)
                
                logits = self.model(X_batch)
    
                loss = self.loss_function(logits, Y_batch)

                self.optimizer.zero_grad()
                loss.backward()

                self.optimizer.step()

                train_loss += loss.item() * len(X_batch)
            
            train_losses.append(train_loss / len(train_loader.dataset))

            if val_loader is not None:
                self.model.eval()
                
                val_loss = 0.0
                val_acc = 0.0
                for batch in val_loader:
                    X_batch, Y_batch = self.prepare_batch(batch)
                    
                    with torch.no_grad():
                        logits = self.model(X_batch)
                        loss = self.loss_function(logits, Y_batch)

                        val_loss += loss.item() * len(X_batch)
                        val_acc += accuracy(self.model.predict(X_batch), Y_batch) * len(X_batch)
                
                val_losses.append(val_loss / len(val_loader.dataset))
                val_accs.append(val_acc / len(val_loader.dataset))
        
        display(pd.DataFrame({
            'train_loss': train_losses,
            'val_loss': val_losses,
            'val_accuracy': val_accs
        }))

## Load Train/Val Split

In [6]:
data = torch.load('../data/train_val.pt')

X_train, X_val = data['X_train'], data['X_val']
Y_train = data['Y_train'].float().reshape(-1, 1)
Y_val = data['Y_val'].float().reshape(-1, 1)

X_train.shape, X_val.shape

(torch.Size([713, 20]), torch.Size([178, 20]))

In [7]:
majority = Y_train.mean().round()
baseline_acc = (Y_val == majority).float().mean()

f'Majority-class baseline: {baseline_acc.item():.2%}'

'Majority-class baseline: 61.80%'

In [8]:
torch.manual_seed(42)

dataset = Dataset(
    X_train,
    Y_train,
    X_val,
    Y_val,
    batch_size=32
)

model = LinearClassifier(in_features=X_train.shape[1])

loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.SGD(model.parameters(), lr=0.01)

trainer = Trainer(
    loss_fn,
    optim,
    max_epochs=500
)

trainer.fit(model, dataset)

,train_loss,val_loss,val_accuracy
0,0.757688,0.746058,0.382022
1,0.716722,0.707933,0.561798
2,0.685157,0.678027,0.674157
3,0.660433,0.654069,0.691011
4,0.640596,0.636416,0.696629
...,...,...,...
495,0.425923,0.467304,0.775281
496,0.425935,0.467482,0.775281
497,0.425856,0.467530,0.780899
498,0.425896,0.467567,0.780899
